# Notebook 19: LangGraph - Advanced Multi-Agent Workflows

**🎯 Goal:** To build on the multi-agent concepts from Notebook 13 by implementing more sophisticated collaboration patterns, including human-in-the-loop and a multi-round debate with a consensus mechanism.

## 🧩 Concept Overview

Real-world problems often require more than just a linear handoff between agents. They require feedback, human oversight, and mechanisms for resolving disagreements. In this notebook, we'll explore two powerful patterns:

1.  **Human-in-the-Loop:** How to pause a workflow and ask for human input before proceeding.
2.  **Debate & Consensus:** How to have multiple agents 'debate' a topic and have a supervisor agent reach a final conclusion based on their arguments.

### 4.1. Recap: Multi-Agent Collaboration

As a reminder from Notebook 13, the core idea of multi-agent workflows is to have a **supervisor** or **router** node that directs tasks to specialized agent nodes based on the current state. This creates a collaborative team of AIs.

### 4.2. Human-in-the-Loop

Sometimes, you need an agent to stop and ask for permission or feedback. We can achieve this by adding a special `human` node and using LangGraph's built-in support for interrupting execution.

The graph will run until it hits the `human` node, then it will pause. We can then resume it with the human's input.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

class HumanInputState(TypedDict):
    work_summary: str
    human_feedback: str

def agent_proposes_work(state: HumanInputState) -> dict:
    print("--- Agent: Proposing work ---")
    return {"work_summary": "I will draft a report on AI ethics."}

def human_approval_node(state: HumanInputState) -> dict:
    print("--- Waiting for Human Approval ---")
    # In a real app, this would be where you get input from a user interface
    # For this example, we'll just check if feedback was manually added.
    if not state.get("human_feedback"):
        print("Pausing for feedback...")
    return {}

def final_work_node(state: HumanInputState) -> dict:
    print("--- Agent: Performing final work ---")
    return {"work_summary": state['work_summary'] + f" based on feedback: '{state['human_feedback']}'"}

# Define the graph
human_workflow = StateGraph(HumanInputState)
human_workflow.add_node("propose", agent_proposes_work)
human_workflow.add_node("human", human_approval_node)
human_workflow.add_node("finalize", final_work_node)

human_workflow.set_entry_point("propose")
human_workflow.add_edge("propose", "human")
human_workflow.add_edge("human", "finalize")
human_workflow.add_edge("finalize", END)

# Compile the graph with interrupt support
app = human_workflow.compile(interrupt_before=["human"])


In [ ]:
# Run the graph until it interrupts
initial_state = {"work_summary": "", "human_feedback": ""}
interrupted_state = app.invoke(initial_state)

print(f"\nGraph paused. Current state: {interrupted_state}")

# Now, provide the human feedback and resume the graph
feedback = "Please focus on the impact on jobs."
resumed_state = app.invoke(interrupted_state, {"human_feedback": feedback})

print(f"\nGraph resumed. Final state: {resumed_state}")

### 4.3. Debate & Consensus

Let's create a more advanced multi-agent pattern. Two agents will 'debate' a topic for a few rounds, and a third 'judge' agent will declare a winner.

This pattern is useful for exploring a problem from multiple perspectives before making a decision.

In [ ]:
class DebateState(TypedDict):
    topic: str
    arguments: Annotated[list, operator.add]
    turn_count: int
    next_speaker: str
    winner: str

def agent_one(state: DebateState) -> dict:
    print(f"--- Agent One (For {state['topic']}) ---")
    return {"arguments": ["Agent One: 'The sky is blue because of Rayleigh scattering.'"]}

def agent_two(state: DebateState) -> dict:
    print(f"--- Agent Two (Against {state['topic']}) ---")
    return {"arguments": ["Agent Two: 'While true, we must also consider the psychological perception of color.'"]}

def judge(state: DebateState) -> dict:
    print("--- Judge ---")
    # A real judge would use an LLM to evaluate the arguments
    return {"winner": "Agent One"}

def debate_router(state: DebateState) -> str:
    # If we have a winner, end
    if state.get("winner"):
        return "end"
    # After a few turns, send to the judge
    if state['turn_count'] >= 2:
        return "judge"
    # Otherwise, continue the debate
    return state['next_speaker']

In [ ]:
debate_workflow = StateGraph(DebateState)
debate_workflow.add_node("agent_one", agent_one)
debate_workflow.add_node("agent_two", agent_two)
debate_workflow.add_node("judge", judge)

# The entry point will now be the router itself
debate_workflow.set_entry_point(debate_router)

debate_workflow.add_conditional_edges(
    debate_router,
    lambda x: x, # The router function itself returns the next node's name
    {
        "agent_one": "agent_one",
        "agent_two": "agent_two",
        "judge": "judge",
        "end": END
    }
)

# After each agent speaks, update state and go back to the router
debate_workflow.add_edge("agent_one", debate_router)
debate_workflow.add_edge("agent_two", debate_router)
debate_workflow.add_edge("judge", debate_router) # The judge's decision also goes to the router

debate_app = debate_workflow.compile()

# Let's manually step through the debate for clarity
s1 = debate_app.invoke({"topic": "Why is the sky blue?", "arguments": [], "turn_count": 0, "next_speaker": "agent_one"})
s2 = debate_app.invoke({"topic": "Why is the sky blue?", "arguments": s1['arguments'], "turn_count": 1, "next_speaker": "agent_two"})
s3 = debate_app.invoke({"topic": "Why is the sky blue?", "arguments": s2['arguments'], "turn_count": 2, "next_speaker": ""})

print(f"\n--- Debate Over ---\nWinner: {s3['winner']}")
print(f"Final Arguments: {s3['arguments']}")

## 🏁 Summary

You've now learned how to orchestrate more complex and realistic multi-agent systems.

1.  **Human-in-the-Loop is Crucial for Control:** By using `interrupt_before`, you can pause your graph to get critical human feedback, ensuring safety and alignment.
2.  **Debate and Consensus Drive Deeper Analysis:** Pitting agents against each other and having a judge evaluate their outputs is a powerful way to explore a problem space and arrive at a more robust conclusion.

In the next notebook, we'll cover even more advanced control flow, like managing loops safely and running nodes in parallel.